# TIGER Beauty Colab runner

Use this notebook from VS Code with the Google Colab kernel, or open it directly in Colab. The notebook clones the GitHub branch into the remote Colab runtime, installs dependencies, persists datasets/runs in Google Drive, and launches experiments from normal Python files in `src/`.

Codex should edit the repo files (`src/*.py`, scripts, report). This notebook should stay a thin runner.

In [ ]:
# Runtime configuration.
REPO_URL = "https://github.com/MayaLager/TIGER-modi.git"
BRANCH = "ksusha/errors-fix"
PROJECT_DIR = "/content/TIGER-modi"

# Keep large data and run JSONs across Colab sessions.
USE_DRIVE_CACHE = True
DRIVE_ROOT = "/content/drive/MyDrive/TIGER-modi-colab"

# Short comparable runs for the report draft. For final paper-style runs,
# increase epochs and use d_model=128, num_layers=4, num_beams=30.
EPOCHS = 5
BATCH_SIZE = 256
D_MODEL = 64
NUM_LAYERS = 2
NUM_BEAMS = 10
DEVICE = "cuda"

In [ ]:
# Check runtime hardware. In Colab: Runtime -> Change runtime type -> GPU.
!nvidia-smi || true
import torch
print("torch", torch.__version__)
print("cuda available", torch.cuda.is_available())

In [ ]:
# Clone or update the experiment branch inside the Colab VM.
import os, subprocess

def run(cmd, cwd=None):
    print("$", cmd)
    subprocess.run(cmd, shell=True, cwd=cwd, check=True)

if not os.path.exists(PROJECT_DIR):
    run(f"git clone --branch {BRANCH} {REPO_URL} {PROJECT_DIR}")
else:
    run("git fetch origin", cwd=PROJECT_DIR)
    run(f"git checkout {BRANCH}", cwd=PROJECT_DIR)
    run("git pull --ff-only", cwd=PROJECT_DIR)

%cd {PROJECT_DIR}
!git status --short --branch
!git log --oneline -3

In [ ]:
# Optional persistent cache in Google Drive. This prevents re-downloading data
# and re-building Sentence-T5 embeddings after Colab restarts.
import os, pathlib, shutil

if USE_DRIVE_CACHE:
    from google.colab import drive
    drive.mount('/content/drive')
    root = pathlib.Path(DRIVE_ROOT)
    for sub in ["data/raw", "data/processed", "runs"]:
        (root / sub).mkdir(parents=True, exist_ok=True)
    pathlib.Path("data").mkdir(exist_ok=True)
    for local, target in [
        ("data/raw", root / "data/raw"),
        ("data/processed", root / "data/processed"),
        ("runs", root / "runs"),
    ]:
        if os.path.islink(local):
            os.unlink(local)
        elif os.path.exists(local):
            shutil.rmtree(local)
        os.symlink(target, local)

!ls -lah data runs

In [ ]:
# Install dependencies. Colab already has PyTorch CUDA; requirements installs
# sentence-transformers, transformers, sklearn, etc.
!python -m pip install -q -r requirements.txt
!python - <<'PY'
import torch, transformers, sklearn
print('torch', torch.__version__)
print('transformers', transformers.__version__)
print('sklearn', sklearn.__version__)
print('cuda', torch.cuda.is_available())
PY

## Data and Semantic IDs

Run these once per cache. Sentence-T5 embeddings and code files are saved under `data/processed/Beauty`.

In [ ]:
# Download and preprocess Amazon Beauty 5-core.
!python src/data/download.py --category Beauty --out data/raw --review-set 5core
!python src/data/preprocess.py --category Beauty --raw data/raw --out data/processed/Beauty --kcore 5
!cat data/processed/Beauty/meta.json

In [ ]:
# RQ-VAE Semantic IDs: paper-shaped L=3, K=256.
!python src/train_semantic_ids.py \
  --data data/processed/Beauty \
  --id-method rqvae \
  --num-levels 3 \
  --codebook-size 256 \
  --rqvae-epochs 200 \
  --device {DEVICE}

In [ ]:
# RQ1 alternative: hierarchical k-means Semantic IDs.
!python src/train_semantic_ids.py \
  --data data/processed/Beauty \
  --id-method hkmeans \
  --num-levels 3 \
  --codebook-size 256 \
  --device {DEVICE}

## Baselines and Ablations

These are short comparable runs. For final runs, set `EPOCHS=200`, `D_MODEL=128`, `NUM_LAYERS=4`, `NUM_BEAMS=30`, then rerun the cells.

In [ ]:
# SASRec baseline.
!python src/train_sasrec.py \
  --data data/processed/Beauty \
  --epochs {EPOCHS} \
  --batch-size {BATCH_SIZE} \
  --device {DEVICE} \
  --out runs/beauty_sasrec_e{EPOCHS}.json

In [ ]:
# TIGER original baseline.
!python src/train_tiger.py \
  --data data/processed/Beauty \
  --codes data/processed/Beauty/codes_rqvae_L3_K256.npy \
  --codebook-size 256 \
  --epochs {EPOCHS} \
  --batch-size {BATCH_SIZE} \
  --d-model {D_MODEL} \
  --num-layers {NUM_LAYERS} \
  --num-beams {NUM_BEAMS} \
  --device {DEVICE} \
  --out runs/beauty_tiger_orig_e{EPOCHS}.json

In [ ]:
# RQ1: TIGER with hierarchical k-means IDs.
!python src/train_tiger.py \
  --data data/processed/Beauty \
  --codes data/processed/Beauty/codes_hkmeans_L3_K256.npy \
  --codebook-size 256 \
  --epochs {EPOCHS} \
  --batch-size {BATCH_SIZE} \
  --d-model {D_MODEL} \
  --num-layers {NUM_LAYERS} \
  --num-beams {NUM_BEAMS} \
  --device {DEVICE} \
  --out runs/beauty_tiger_hkmeans_e{EPOCHS}.json

In [ ]:
# RQ2: token order and collision-token placement.
!python src/train_tiger.py --data data/processed/Beauty --codes data/processed/Beauty/codes_rqvae_L3_K256.npy --codebook-size 256 --token-order reversed --epochs {EPOCHS} --batch-size {BATCH_SIZE} --d-model {D_MODEL} --num-layers {NUM_LAYERS} --num-beams {NUM_BEAMS} --device {DEVICE} --out runs/beauty_tiger_reversed_e{EPOCHS}.json
!python src/train_tiger.py --data data/processed/Beauty --codes data/processed/Beauty/codes_rqvae_L3_K256.npy --codebook-size 256 --token-order permuted --epochs {EPOCHS} --batch-size {BATCH_SIZE} --d-model {D_MODEL} --num-layers {NUM_LAYERS} --num-beams {NUM_BEAMS} --device {DEVICE} --out runs/beauty_tiger_permuted_e{EPOCHS}.json
!python src/train_tiger.py --data data/processed/Beauty --codes data/processed/Beauty/codes_rqvae_L3_K256.npy --codebook-size 256 --collision-first --epochs {EPOCHS} --batch-size {BATCH_SIZE} --d-model {D_MODEL} --num-layers {NUM_LAYERS} --num-beams {NUM_BEAMS} --device {DEVICE} --out runs/beauty_tiger_collision_first_e{EPOCHS}.json

In [ ]:
# RQ4: non-autoregressive baselines. Use eval cap first; full eval can be slow.
EVAL_USERS_NAR = 1000
!python src/train_nar.py --data data/processed/Beauty --codes data/processed/Beauty/codes_rqvae_L3_K256.npy --codebook-size 256 --model parallel --epochs {EPOCHS} --batch-size {BATCH_SIZE} --d-model {D_MODEL} --num-layers {NUM_LAYERS} --num-beams {NUM_BEAMS} --eval-users {EVAL_USERS_NAR} --device {DEVICE} --out runs/beauty_nar_parallel_e{EPOCHS}_eval{EVAL_USERS_NAR}.json
!python src/train_nar.py --data data/processed/Beauty --codes data/processed/Beauty/codes_rqvae_L3_K256.npy --codebook-size 256 --model denoiser --epochs {EPOCHS} --batch-size {BATCH_SIZE} --d-model {D_MODEL} --num-layers {NUM_LAYERS} --num-beams {NUM_BEAMS} --eval-users {EVAL_USERS_NAR} --device {DEVICE} --out runs/beauty_nar_denoiser_e{EPOCHS}_eval{EVAL_USERS_NAR}.json

In [ ]:
# Summarize all available Beauty run JSONs.
import glob, json, os
rows = []
for path in sorted(glob.glob("runs/beauty_*.json")):
    d = json.load(open(path))
    test = d.get("test", {})
    cfg = d.get("config", {})
    rows.append({
        "run": os.path.basename(path),
        "eval": "full" if cfg.get("eval_users") in (None, 0) else f"cap {cfg.get('eval_users')}",
        "R@10": test.get("recall@10"),
        "N@10": test.get("ndcg@10"),
        "invalid": test.get("invalid_code_rate"),
        "collision": test.get("collision_rate_pre_disambiguation"),
        "best_val_R@10": d.get("best_val_recall@10"),
    })

try:
    import pandas as pd
    display(pd.DataFrame(rows))
except Exception:
    for r in rows:
        print(r)